# Train the on-device fingerspelling model (ASL & FSL)

This notebook runs the offline pipeline that produces `public/models/letters-<lang>.json`
for the sign language learning app. The model is a small MLP over MediaPipe hand
landmarks; at runtime the app runs the forward pass in plain TypeScript, so **no ML
runtime ships to the browser**.

**Pipeline:** download dataset -> clean (dedupe + group) -> extract landmarks -> train -> export JSON.

Datasets used:
- ASL: [`grassknoted/asl-alphabet`](https://www.kaggle.com/datasets/grassknoted/asl-alphabet)
- FSL: [`japorton/fsl-dataset`](https://www.kaggle.com/datasets/japorton/fsl-dataset)

Set the **Configuration** cell, then run all. A CPU runtime is enough.

In [ ]:
# Configuration ------------------------------------------------------------
REPO_URL = "https://github.com/Wisloo/SignLanguageApplication.git"

LANG = "asl"  # "asl" or "fsl"

DATASETS = {
    "asl": "grassknoted/asl-alphabet",
    "fsl": "japorton/fsl-dataset",
}

assert LANG in DATASETS, "LANG must be 'asl' or 'fsl'"

In [ ]:
# Keep the session connected during the long extraction -------------------
# Colab's free tier can drop an idle/running session. This nudges the
# frontend to reconnect about once a minute. Keep the browser tab open.
from IPython.display import display, Javascript

display(
    Javascript(
        r'''
function keepAlive() {
  const btn = document.querySelector('colab-connect-button');
  if (!btn) return;
  const root = btn.shadowRoot || btn;
  const label = (root.textContent || '').trim();
  if (label.startsWith('Connect')) {
    btn.click();
  }
}
setInterval(keepAlive, 60000);
'''
    )
)

In [ ]:
# Clone the repository and install dependencies ---------------------------
import os
import subprocess
import sys

if not os.path.isdir("SignLanguageApplication"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
os.chdir("SignLanguageApplication")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "mediapipe", "pillow", "kagglehub"],
    check=True,
)
print("Repo ready at", os.getcwd())

In [ ]:
# Download the dataset via kagglehub --------------------------------------
# Public datasets download without credentials; if prompted, add a Kaggle
# API token (kagglehub reads it automatically).
import kagglehub

dataset_root = kagglehub.dataset_download(DATASETS[LANG])
print("Downloaded to", dataset_root)

### Resolve the layout

Both datasets are normally folder-per-class (`A/`, `B/`, ...). If a dataset
arrives as a flat dump of images, we best-effort regroup it by the first
character of each filename.

In [ ]:
# Find (or build) a folder-per-class directory ----------------------------
from pathlib import Path


def find_class_root(base):
    """Locate the directory whose children are single-letter class folders."""
    for path in sorted(Path(base).rglob("*")):
        if not path.is_dir():
            continue
        letters = [
            p.name
            for p in path.iterdir()
            if p.is_dir() and len(p.name) == 1 and p.name.isalpha()
        ]
        if len(letters) >= 20:
            return path
    return None


def reorganize_flat(base, out, min_per_letter=100):
    """Group a flat image dump into per-letter folders via filename prefix."""
    out = Path(out)
    counts = {}
    for image in Path(base).rglob("*"):
        if image.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        letter = image.stem[0].upper()
        if not letter.isalpha():
            continue
        target = out / letter
        target.mkdir(parents=True, exist_ok=True)
        link = target / image.name
        if not link.exists():
            link.symlink_to(image.resolve())
        counts[letter] = counts.get(letter, 0) + 1
    return {k: v for k, v in counts.items() if v >= min_per_letter}


resolved = find_class_root(dataset_root)
if resolved is None:
    print("Flat layout detected; regrouping by filename prefix...")
    resolved = Path("data") / f"{LANG}_classes"
    kept = reorganize_flat(dataset_root, resolved)
    print("Letters found:", sorted(kept))

CLASS_ROOT = str(resolved)
print("CLASS_ROOT =", CLASS_ROOT)

In [ ]:
# Clean: integrity check, exact-dupe removal, near-duplicate grouping -----
subprocess.run(
    [
        sys.executable,
        "scripts/train/clean_images.py",
        "--dataset",
        CLASS_ROOT,
        "--out",
        f"data/{LANG}_manifest.json",
    ],
    check=True,
)

In [ ]:
# Extract landmarks -------------------------------------------------------
# ASL ships `space`/`del` hand gestures - use them for the 'none' class.
# (The `nothing` folder has no hand and is dropped by the extractor.)
negative_dirs = [
    p
    for p in (Path(CLASS_ROOT) / "space", Path(CLASS_ROOT) / "del")
    if p.is_dir()
]

cmd = [
    sys.executable,
    "scripts/train/extract_landmarks.py",
    "--dataset",
    CLASS_ROOT,
    "--manifest",
    f"data/{LANG}_manifest.json",
    "--out",
    f"data/{LANG}_landmarks.npz",
]
if negative_dirs:
    cmd += ["--negatives", *[str(p) for p in negative_dirs]]
subprocess.run(cmd, check=True)

In [ ]:
# Train and export --------------------------------------------------------
subprocess.run(
    [
        sys.executable,
        "scripts/train/train_mlp.py",
        "--npz",
        f"data/{LANG}_landmarks.npz",
        "--lang",
        LANG,
        "--class-weight",
    ],
    check=True,
)

### (optional) Build the handshape reference images

Picks one clean frame per letter from the dataset for the learn-page pictures.
The repo already ships these, so rerun only when you change the selection.

In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/train/make_reference_images.py",
        "--dataset",
        CLASS_ROOT,
        "--out",
        f"public/signs/{LANG}",
    ],
    check=True,
)

import shutil

shutil.make_archive(f"{LANG}_signs", "zip", f"public/signs/{LANG}")
from google.colab import files as colab_files

colab_files.download(f"{LANG}_signs.zip")

In [ ]:
# Download the model ------------------------------------------------------
# Drop the file at public/models/letters-<lang>.json in your local checkout.
from google.colab import files

model_path = f"public/models/letters-{LANG}.json"
print("Model size:", os.path.getsize(model_path), "bytes")
files.download(model_path)

### Notes

- The exported JSON is validated on load against `NORMALIZATION_ID`, so keep
  `scripts/train/*.py` and `src/lib/vision/normalize.ts` in sync if you change
  the landmark transform.
- J and Z are motion signs and are intentionally excluded; the app grades them
  with its trajectory recogniser.
- Watch the confusion matrix and the 'suspected mislabels' list - the datasets
  contain some wrong labels and near-identical frames.